# Rec Systems III
# User Based Collaborative Filtering

This workbook is part of a series, listed below:
- <a href="https://nbviewer.org/github/pw598/Articles/blob/main/Rec%20Systems%20I%20-%20Baseline%20Methods.ipynb">Rec Systems I - Baseline Methods</a>
- <a href="https://nbviewer.org/github/pw598/Articles/blob/main/Rec%20Systems%20II%20-%20Content%20Based.ipynb">Rec Systems II - Content Based</a>
- <a href="https://nbviewer.org/github/pw598/Articles/blob/main/Rec%20Systems%20III%20-%20User-Based%20Collaborative%20Filtering.ipynb">Rec Systems III - User-Based Collaborative Filtering</a>
- <a href="https://nbviewer.org/github/pw598/Articles/blob/main/Rec%20Systems%20IV%20-%20Item-Based%20Collaborative%20Filtering.ipynb">Rec Systems IV - Item-Based Collaborative Filtering</a>
- <a href="https://nbviewer.org/github/pw598/Articles/blob/main/Rec%20Systems%20V%20-%20Matrix%20Factorization.ipynb">Rec Systems V - Matrix Factorization</a>

In the previous two notebooks we have explored baseline recommendation methods, and content (description) based recommendation. In this workbook, we'll take a look at collaborative filtering methods, which measure the similarity between users or items in order to make weighted or otherwise neighborhood-based predictions.

First, I'll import necessary libraries and the movielens dataset.

In [1]:
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import cosine_similarity

import warnings 
warnings.filterwarnings('ignore')

In [2]:
movie_info = pd.read_csv('movielens_data/movie_info.csv',header=0,sep=",")
movie_info.head(1)

,movie id,movie title,release date,unknown,Action,Adventure,Animation,Children's,Comedy,Crime,...,Fantasy,Film-Noir,Horror,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,1,Toy Story (1995),01-Jan-95,0,0,0,1,1,1,0,...,0,0,0,0,0,0,0,0,0,0


In [3]:
ratings = pd.read_csv('movielens_data/ratings.csv',header=0,sep=",")
ratings.drop(columns='unix_timestamp', inplace=True)
ratings.head(1)

,user_id,movie_id,rating
0,196,242,3


In [4]:
user_info = pd.read_csv('movielens_data/user_demographics.csv',header=0,sep=",")
user_info.head(1)

,user_id,age,sex,occupation,zip_code
0,1,24,M,technician,85711


# Error Metrics

I'll choose RMSE as an error metric, partly to remain consistent with the prior notebooks.

In [5]:
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

# Train/Test Split

I'll set aside 25% of the data for testing.

In [6]:
X = ratings.copy()
X_train, X_test = train_test_split(X, test_size = 0.25, random_state=123)

# User-Based Collaborative Filtering

In this context, user-based collaborative filtering means that, given a movie and user to predict for, we first determine the similarity between that user and other users, and then choose a rating based on the ratings of similar users for that movie. In this case, I'll use a weighted sum, and base it on cosine similarity, though one can use other methods such as Pearson correlation.

In [7]:
r_matrix = X_train.pivot_table(values='rating', index='user_id', columns='movie_id')
r_matrix.head()

movie_id,1,2,3,4,5,6,7,8,9,10,...,1672,1673,1674,1675,1676,1677,1678,1679,1680,1681
user_id,,,,,,,,,,,,,,,,,,,,,
1,NaN,NaN,NaN,3.0,NaN,5.0,4.0,1.0,5.0,3.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


The movie by user ratings matrix is clearly very sparse. Next, I'll create the cosine similarity matrix between users.

In [8]:
r_matrix_dummy = r_matrix.copy().fillna(0)
cosine_sim = cosine_similarity(r_matrix_dummy, r_matrix_dummy)
cosine_sim = pd.DataFrame(cosine_sim, index=r_matrix.index, columns=r_matrix.index)
cosine_sim.head(5)

user_id,1,2,3,4,5,6,7,8,9,10,...,934,935,936,937,938,939,940,941,942,943
user_id,,,,,,,,,,,,,,,,,,,,,
1,1.000000,0.105270,0.037687,0.038502,0.243338,0.311555,0.329942,0.255862,0.056152,0.246458,...,0.254688,0.043090,0.239825,0.172872,0.107195,0.096027,0.266754,0.105442,0.120018,0.286551
2,0.105270,1.000000,0.063560,0.151517,0.084214,0.209577,0.100226,0.094336,0.197575,0.114654,...,0.119047,0.283104,0.296910,0.346577,0.274295,0.173581,0.213554,0.167879,0.130507,0.132538
3,0.037687,0.063560,1.000000,0.201881,0.000000,0.025452,0.044646,0.074497,0.000000,0.061825,...,0.029908,0.058906,0.133683,0.026763,0.094238,0.018583,0.113169,0.060060,0.057123,0.021559
4,0.038502,0.151517,0.201881,1.000000,0.043530,0.065006,0.092797,0.168242,0.152810,0.038116,...,0.040740,0.050151,0.098714,0.191397,0.067213,0.000000,0.122321,0.061360,0.183415,0.060571
5,0.243338,0.084214,0.000000,0.043530,1.000000,0.187913,0.303321,0.171243,0.065243,0.133386,...,0.253168,0.059017,0.075344,0.099813,0.103833,0.062922,0.189139,0.131154,0.104918,0.249283


Then the prediction function.

In [9]:
def predict_usercf(movie, user):
    dummy = r_matrix[movie].fillna(0)
    sim_scores = cosine_sim[user]
    pred = sum(sim_scores * dummy) / sim_scores[dummy > 0].sum()
    return pred

### Prediction Upon Training Set

In [10]:
p_matrix = r_matrix.copy()

In [11]:
for movie in tqdm(r_matrix.columns):
    for user in r_matrix.index:
        if not pd.isnull(r_matrix[movie][user]):
            pred = predict_usercf(movie, user)
            p_matrix[movie][user] = pred

100%|██████████| 1641/1641 [01:34<00:00, 17.31it/s]


In [12]:
preds = []
actuals = []

for movie in tqdm(p_matrix.columns):
    for user in p_matrix.index:
        if not pd.isnull(p_matrix[movie][user]):
            preds.append(p_matrix[movie][user])
            actuals.append(r_matrix[movie][user])

100%|██████████| 1641/1641 [00:15<00:00, 105.47it/s]


The RMSE on the training set is as follows:

In [13]:
rmse(actuals, preds)

0.9305195802428011

### Prediction Upon Test Set

Next, we'll predict upon the test set.

In [14]:
r_matrix_test = X_test.pivot_table(values='rating', index='user_id', columns='movie_id')
r_matrix_test.head(2)

movie_id,1,2,3,4,5,6,7,8,9,10,...,1652,1653,1654,1655,1658,1659,1662,1664,1666,1682
user_id,,,,,,,,,,,,,,,,,,,,,
1,5.0,3.0,4.0,NaN,3.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [15]:
print(r_matrix.shape)
print(r_matrix_test.shape)

(943, 1641)
(943, 1454)


In [16]:
p_matrix = r_matrix_test.copy()

for movie in tqdm(r_matrix_test.columns):
    for user in r_matrix_test.index:
        if not pd.isnull(r_matrix_test[movie][user]):
            try:
                pred = predict_usercf(movie, user)
                p_matrix[movie][user] = pred
            except:
                pass

100%|██████████| 1454/1454 [00:40<00:00, 36.03it/s] 


In [17]:
preds = []
actuals = []

for movie in tqdm(p_matrix.columns):
    for user in p_matrix.index:
        if not pd.isnull(p_matrix[movie][user]):
            preds.append(p_matrix[movie][user])
            actuals.append(r_matrix_test[movie][user])

100%|██████████| 1454/1454 [00:14<00:00, 101.04it/s]


The test RMSE is as follows:

In [18]:
rmse(actuals, preds)

1.0214503777664212

For context, the performance of baseline methods was found to be as follows:

In [19]:
results_df = pd.DataFrame({"model":['Simple Average', 'Naive User-Based Avg', 'Naive Item-Based Avg', 'Vote-Weighted Avg', 'IMDB Weighted Avg'],
                        "train_rmse":[1.123, 1.025, 0.995, 1.123, 1.036], 
                        "test_rmse":[1.134, 1.054, 1.028, 1.134, 1.052]})

results_df

,model,train_rmse,test_rmse
0,Simple Average,1.123,1.134
1,Naive User-Based Avg,1.025,1.054
2,Naive Item-Based Avg,0.995,1.028
3,Vote-Weighted Avg,1.123,1.134
4,IMDB Weighted Avg,1.036,1.052


I'll append these new results.

In [20]:
def append_results_df(model, train_rmse, test_rmse):
    new_row = len(results_df) + 1
    results_df.loc[new_row,'model'] = model
    results_df.loc[new_row,'train_rmse'] = round(train_rmse,3)
    results_df.loc[new_row,'test_rmse'] = round(test_rmse,3)

In [36]:
append_results_df('User-Based CF', 0.931, 1.02)
results_df

,model,train_rmse,test_rmse
0,Simple Average,1.123,1.134
1,Naive User-Based Avg,1.025,1.054
2,Naive Item-Based Avg,0.995,1.028
3,Vote-Weighted Avg,1.123,1.134
4,IMDB Weighted Avg,1.036,1.052
6,User-Based CF,0.931,1.020


The next workbook in the series looks at item-based collaborative filtering, and can be found <a href="https://nbviewer.org/github/pw598/Articles/blob/main/Rec%20Systems%20IV%20-%20Item-Based%20Collaborative%20Filtering.ipynb">here</a>.